In [2]:

%load_ext autoreload
%autoreload 2
from game_infos import get_game_info
from league_infos import get_league_urls, get_game_urls
import pandas as pd
from init_db import create_db_engine
from tqdm import tqdm
import uuid
engine = create_db_engine()
root = 'https://www.handball.net/ligen?organization=Hamburg'
league_urls = get_league_urls(root)

In [ ]:
current_status = pd.read_sql_table('game_details', engine)
hamburg_data = pd.DataFrame()
for league_url in tqdm(league_urls, position=0, leave=True):
    games_urls = get_game_urls(league_url)
    game_data_list = []
    
    
    game_ids = current_status['game_id'].unique()
    games_urls_filtered = [game_url for game_url in games_urls if game_url.split('/')[-1] not in game_ids]
    for game_url in tqdm(games_urls_filtered, position=0, leave=True):
        game_info = get_game_info(game_url)
        game_data_list.append(game_info)
        if game_info.empty:
            break

    hamburg_data = pd.concat([hamburg_data, pd.concat(game_data_list)], ignore_index=True)

100%|██████████| 20/20 [03:42<00:00, 11.13s/it]


In [16]:
hamburg_data = pd.read_csv('hamburg_data.csv')
uuid_mapping = {team: str(uuid.uuid4()) for team in hamburg_data['team'].unique()}
hamburg_data['team_id'] = hamburg_data['team'].map(uuid_mapping)
hamburg_data["team_name"] = hamburg_data["team"]
hamburg_data[hamburg_data["team"] == "HSG Pinnau"]

,number,name,goals,two_min,card,team,date,game_id,league_id,league_name,team_id,team_name
1625,1,Matei-Romeo Ionita,0,0,NaN,HSG Pinnau,2024-09-15 17:30:00,handball4all.hamburg.7843416,handball4all.hamburg.126986,Hamburger HV - Männer Landesliga (Gruppe 1),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau
1626,4,Dominik Stolz,5,0,yellow,HSG Pinnau,2024-09-15 17:30:00,handball4all.hamburg.7843416,handball4all.hamburg.126986,Hamburger HV - Männer Landesliga (Gruppe 1),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau
1627,6,Marco Pucks,0,1,NaN,HSG Pinnau,2024-09-15 17:30:00,handball4all.hamburg.7843416,handball4all.hamburg.126986,Hamburger HV - Männer Landesliga (Gruppe 1),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau
1628,10,Kevin Krauel,0,0,NaN,HSG Pinnau,2024-09-15 17:30:00,handball4all.hamburg.7843416,handball4all.hamburg.126986,Hamburger HV - Männer Landesliga (Gruppe 1),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau
1629,11,Julian Stolz,5,0,NaN,HSG Pinnau,2024-09-15 17:30:00,handball4all.hamburg.7843416,handball4all.hamburg.126986,Hamburger HV - Männer Landesliga (Gruppe 1),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau
...,...,...,...,...,...,...,...,...,...,...,...,...
12508,17,Sandra Guillery,6,0,NaN,HSG Pinnau,2024-12-07 15:30:00,handball4all.hamburg.7839581,handball4all.hamburg.126896,Frauen Oberliga (200),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau
12509,21,Nathalie Grünwald,0,0,NaN,HSG Pinnau,2024-12-07 15:30:00,handball4all.hamburg.7839581,handball4all.hamburg.126896,Frauen Oberliga (200),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau
12510,22,Rebecca Fahl,1,0,NaN,HSG Pinnau,2024-12-07 15:30:00,handball4all.hamburg.7839581,handball4all.hamburg.126896,Frauen Oberliga (200),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau
12511,23,Natalie Engelmann,6,0,NaN,HSG Pinnau,2024-12-07 15:30:00,handball4all.hamburg.7839581,handball4all.hamburg.126896,Frauen Oberliga (200),796f7762-eb5b-4160-b85e-8425621157e9,HSG Pinnau


In [14]:
# Create league table
hamburg_data[["league_id", "league_name"]].drop_duplicates().to_sql('leagues', engine, if_exists='replace', index=False)
# Create Team Table
uuid_mapping = {team: str(uuid.uuid4()) for team in hamburg_data['team'].unique()}
hamburg_data['team_id'] = hamburg_data['team'].map(uuid_mapping)
hamburg_data["team_name"] = hamburg_data["team"]
hamburg_data[["team_id", "team_name","league_id"]].drop_duplicates().to_sql('teams', engine, if_exists='replace', index=False)

# Create Game Table
hamburg_data["record_id"] = hamburg_data["game_id"].apply(lambda x: uuid.uuid4())
hamburg_data = hamburg_data.drop(columns=["league_name", "team_name", "team"])
hamburg_data.to_sql('game_details', engine, if_exists='replace', index=False)

753

In [61]:
current_status.to_sql('game_details', engine, if_exists='replace', index=False)

753